In [ ]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# 1) Point this at the parent directory that contains AT/, BE/, BG/, etc.
DATA_ROOT = Path('/Users/harshit/Desktop/nature-in-language/data/results/transcript')

records = []

# 2) Walk each country folder
for country_dir in DATA_ROOT.iterdir():
    if not country_dir.is_dir():
        continue
    country = country_dir.name

    # 3) Find all the .csv files in that folder
    for csv_path in country_dir.glob('*.csv'):
        # extract year from filename (assumes it starts with YYYY_)
        m = re.match(r'(\d{4})_', csv_path.name)
        if not m:
            continue
        year = int(m.group(1))

        # 4) Load and flatten all numeric columns
        df = pd.read_csv(csv_path)
        nums = df.select_dtypes(include=[np.number]).to_numpy().flatten()
        mean_val = np.nanmean(nums)  # ignore any NaNs

        records.append({
            'Year': year,
            'Country': country,
            'MeanValue': mean_val
        })

# 5) Build the summary pivot table
summary = (
    pd.DataFrame(records)
      .pivot(index='Year', columns='Country', values='MeanValue')
      .sort_index()               # sort years
      .sort_index(axis=1)         # sort country codes
)

print(summary)

# 6) Save to Excel (or CSV)
summary.to_csv('/Users/harshit/Desktop/nature-in-language/data/results/transcript/Average/mean_by_country_year.csv')
# – or –
# summary.to_csv('mean_by_country_year.csv')

print(summary.shape)        # should be (# distinct years, # distinct countries)
print(summary.head(5))      # peek at the first 5 years
print(summary.isna().sum()) # see if any country/year combos ended up missing

Country        AT        BA        BE        BG        CZ        DK        EE  \
Year                                                                            
1996     0.012347       NaN       NaN       NaN       NaN       NaN       NaN   
1997     0.012372       NaN       NaN       NaN       NaN       NaN       NaN   
1998    -0.002371 -0.016148       NaN       NaN       NaN       NaN       NaN   
1999     0.003298 -0.045875       NaN       NaN       NaN       NaN       NaN   
2000     0.002077 -0.000400       NaN       NaN       NaN       NaN       NaN   
2001    -0.010914 -0.013530       NaN       NaN       NaN       NaN       NaN   
2002     0.021190 -0.014768       NaN       NaN       NaN       NaN       NaN   
2003     0.001870  0.013770       NaN       NaN       NaN       NaN       NaN   
2004     0.012748 -0.022607       NaN       NaN       NaN       NaN       NaN   
2005     0.013373 -0.054391       NaN       NaN       NaN       NaN       NaN   
2006    -0.000024 -0.010702 

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# 1) Load the pivoted CSV (adjust path as needed)
summary = pd.read_csv('/Users/harshit/Desktop/nature-in-language/data/results/transcript/Average/mean_by_country_year.csv', index_col='Year')

# 2) Plot one line per country
plt.figure(figsize=(10, 6), dpi=1200)
for country in summary.columns:
    plt.plot(summary.index, summary[country], marker='o', label=country)

plt.title('Mean Value by Country Over Time')
plt.xlabel('Year')
plt.ylabel('Mean of Numeric Entries')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')  # legend outside
plt.tight_layout()
plt.show()

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# 1) Load the same CSV
summary = pd.read_csv('/Users/harshit/Desktop/nature-in-language/data/results/transcript/Average/mean_by_country_year.csv', index_col='Year')

# 2) Turn the DataFrame into a 2D array and plot
data = summary.T.values  # rows=countries, cols=years

red_green = LinearSegmentedColormap.from_list('red_green', ['red','green'])

plt.figure(figsize=(12, 6), dpi=1200)
plt.imshow(data, origin='lower', aspect='auto', interpolation='nearest', cmap='BuGn')
plt.colorbar(label='Mean Value')

# 3) Label axes
plt.xticks(ticks=np.arange(len(summary.index)), labels=summary.index, rotation=45)
plt.yticks(ticks=np.arange(len(summary.columns)), labels=summary.columns)

plt.title('Heatmap of Mean Values (Years × Countries)')
plt.xlabel('Year')
plt.ylabel('Country Code')
plt.tight_layout()
plt.show()